In [ ]:
# 1. Setup & Data Loading
from pathlib import Path
import pandas as pd

data_dir = Path("../data/processed/features")

BACKTEST_COLS = [
    'timestamp', 'price', 'sign', 'qty',
    'best_bid', 'best_ask', 'midprice', 'toxic'
]

ASSETS = ['BTCUSDT', 'ETHUSDT', 'SOLUSDT']
WEEKS = ['week1', 'week2', 'week3']

BACKTEST_COLS = [
    'timestamp', 'price', 'sign', 'qty',
    'spread', 'midprice', 'toxic'
]

def load_backtest_data(data_dir, asset, week):
    path = Path(data_dir) / f"{asset}_{week}_full_features.parquet"
    df = pd.read_parquet(path, columns=BACKTEST_COLS)
    
    df = df.sort_values('timestamp').reset_index(drop=True)
    df = df.dropna()
    df['timestamp'] = df['timestamp'].astype('int64') / 1e9
    
    # Reconstruct best bid/ask
    df['best_bid'] = df['midprice'] - df['spread'] / 2
    df['best_ask'] = df['midprice'] + df['spread'] / 2
    df['mid'] = df['midprice']  # alias for backtest engine
    
    return df

# Load as dict, not concatenated DataFrame
all_data = {}
for asset in ASSETS:
    for week in WEEKS:
        all_data[(asset, week)] = load_backtest_data(data_dir, asset, week)
        print(f"  {asset} {week}: {len(all_data[(asset, week)]):,} trades")




  BTCUSDT week1: 10,071,946 trades
  BTCUSDT week2: 12,032,260 trades
  BTCUSDT week3: 24,249,845 trades
  ETHUSDT week1: 4,554,411 trades
  ETHUSDT week2: 7,137,925 trades
  ETHUSDT week3: 10,051,489 trades
  SOLUSDT week1: 3,968,322 trades
  SOLUSDT week2: 5,757,031 trades
  SOLUSDT week3: 11,434,844 trades
timestamp    float64
price        float64
sign           int64
qty          float64
spread       float64
midprice     float64
toxic           bool
best_bid     float64
best_ask     float64
mid          float64
dtype: object

      timestamp    price  sign    qty  spread  midprice  toxic  best_bid  best_ask       mid
0  1.725840e+09  54849.9    -1  0.024     0.1  54849.95  False   54849.9   54850.0  54849.95
1  1.725840e+09  54850.0     1  0.050     0.1  54849.95  False   54849.9   54850.0  54849.95
2  1.725840e+09  54850.6     1  0.001     0.1  54849.95  False   54849.9   54850.0  54849.95
3  1.725840e+09  54850.4     1  0.001     0.1  54849.95  False   54849.9   54850.0  54849.95

In [78]:
import numpy as np
import pandas as pd
from dataclasses import dataclass, field
from typing import Optional
import warnings

@dataclass
class MarketMakerState:
    """
    Tracks the MM's position at any point in time.
    cash: realised cash from fills (positive = received cash from sells, negative = paid for buys)
    inventory: net position in base asset (positive = long)
    fills: list of (timestamp, side, price, qty, pnl_after_fill)
    """
    cash: float = 0.0
    inventory: float = 0.0
   

def avellaneda_stoikov_quotes_adjusted(s, q, gamma, sigma, kappa,
                                        T_minus_t, best_bid, best_ask):
    """
    A-S in log-return space throughout.
    
    sigma: log-return vol per bar (dimensionless)
    gamma: risk aversion in log-return space
    q: inventory in base asset units
    """
    # Work in log-price space
    log_s = np.log(s)
    log_best_bid = np.log(best_bid)
    log_best_ask = np.log(best_ask)
    
    # Normalised inventory
    q_normalised = q / s
    
    # Reservation log-price
    log_r = log_s - q_normalised * gamma * (sigma ** 2) * T_minus_t
    
    # Half-spread in log-return space
    inner = max(1 + gamma / kappa, 1e-10)
    delta_log = gamma * (sigma ** 2) * T_minus_t + (2 / gamma) * np.log(inner)
    
    # MM quotes in log-space, anchored to best bid/ask
    log_mm_bid = log_best_bid
    log_mm_ask = log_best_ask
    
    # Step back if reservation price makes quoting unattractive
    if log_r < log_best_bid - delta_log:
        log_mm_bid = log_r - delta_log
    
    if log_r > log_best_ask + delta_log:
        log_mm_ask = log_r + delta_log
    
    # Convert back to price space
    mm_bid = np.exp(log_mm_bid)
    mm_ask = np.exp(log_mm_ask)
    r = np.exp(log_r)
    
    spread_bps = (mm_ask - mm_bid) / s * 10000
    
    return mm_bid, mm_ask, r, delta_log, spread_bps
     
def compute_backtest_metrics(results_df: pd.DataFrame,
                              risk_free_rate: float = 0.0) -> dict:
    # Resample PnL to 1-minute intervals
    r = results_df.set_index(
        pd.to_datetime(results_df['timestamp'], unit='s')
    )['pnl']
    
    pnl_1min = r.resample('1min').last().ffill().dropna()
    pnl_changes = pnl_1min.diff().dropna()
    
    if len(pnl_changes) < 10:
        return {'error': 'insufficient data'}
    
    # Sharpe: annualised using 1-minute returns
    periods_per_year = 1440 * 365
    sharpe = (pnl_changes.mean() / pnl_changes.std()) * np.sqrt(periods_per_year)
    
    # Max drawdown
    cumulative = pnl_1min - pnl_1min.iloc[0]
    rolling_max = cumulative.cummax()
    drawdown = cumulative - rolling_max
    max_drawdown = drawdown.min()
    
    # Fill statistics
    n_buy_fills = results_df['filled_buy'].sum()
    n_sell_fills = results_df['filled_sell'].sum()
    n_fills = n_buy_fills + n_sell_fills
    
    # Fill rate: fraction of bars where MM was filled on at least one side
    fill_rate = (results_df['filled_buy'] | results_df['filled_sell']).mean()
    
    # Competitive rate: fraction of bars where MM is competitive on at least one side
    competitive_rate = (
        results_df['mm_ask_competitive'] | results_df['mm_bid_competitive']
    ).mean()
    
    # Inventory statistics
    inv = results_df['inventory']
    
    return {
        'sharpe': sharpe,
        'max_drawdown': max_drawdown,
        'total_pnl': results_df['pnl'].iloc[-1],
        'fill_rate': fill_rate,
        'n_fills': n_fills,
        'n_buy_fills': n_buy_fills,
        'n_sell_fills': n_sell_fills,
        'competitive_rate': competitive_rate,
        'inventory_mean': inv.mean(),
        'inventory_std': inv.std(),
        'inventory_95th': inv.abs().quantile(0.95),
        'inventory_max': inv.abs().max(),
        'mean_spread_bps': results_df['spread_bps'].mean(),
    }

In [79]:
def aggregate_to_bars(df: pd.DataFrame,
                      bar_seconds: int = 1) -> pd.DataFrame:
    """
    Aggregate trade-level data to fixed time bars.
    Fast version: split by direction before groupby to avoid lambda lookups.
    """
    df = df.copy()
    df['bar'] = (df['timestamp'] // bar_seconds) * bar_seconds
    
    # --- Base aggregation: book state and volume ---
    bars = df.groupby('bar').agg(
        timestamp=('timestamp', 'first'),
        mid=('mid', 'first'),
        best_bid=('best_bid', 'first'),
        best_ask=('best_ask', 'first'),
        bar_high=('price', 'max'),
        bar_low=('price', 'min'),
        bar_volume=('qty', 'sum'),
        bar_trades=('timestamp', 'count'),
        toxic_rate=('toxic', 'mean'),
    ).reset_index()
    
    # --- Buy aggressor trades only ---
    buys = df[df['sign'] == 1].groupby('bar').agg(
        any_buy=('price', 'count'),
        max_buy_price=('price', 'max'),
        buy_volume=('qty', 'sum'),
        buy_trades=('timestamp', 'count'),
    ).reset_index()
    buys['any_buy'] = True
    
    # --- Sell aggressor trades only ---
    sells = df[df['sign'] == -1].groupby('bar').agg(
        any_sell=('price', 'count'),
        min_sell_price=('price', 'min'),
        sell_volume=('qty', 'sum'),
        sell_trades=('timestamp', 'count'),
    ).reset_index()
    sells['any_sell'] = True
    
    # --- Merge ---
    bars = bars.merge(
        buys[['bar', 'any_buy', 'max_buy_price', 'buy_volume', 'buy_trades']], 
        on='bar', how='left'
    )
    bars = bars.merge(
        sells[['bar', 'any_sell', 'min_sell_price', 'sell_volume', 'sell_trades']], 
        on='bar', how='left'
    )
    
    # Fill missing directions with False/NaN
    bars['any_buy'] = bars['any_buy'].fillna(False)
    bars['any_sell'] = bars['any_sell'].fillna(False)
    
    bars = bars.drop(columns='bar').reset_index(drop=True)
    
    return bars

# Test on BTC week 1
btc_w1 = all_data[('BTCUSDT', 'week1')]
btc_bars = aggregate_to_bars(btc_w1, bar_seconds=1)

print(f"Trades: {len(btc_w1):,}")
print(f"1-second bars: {len(btc_bars):,}")
print(f"Mean trades per bar: {btc_bars['bar_trades'].mean():.1f}")
print(f"Bars with any buy: {btc_bars['any_buy'].mean():.1%}")
print(f"Bars with any sell: {btc_bars['any_sell'].mean():.1%}")
print(f"\nSample:")
print(btc_bars.head(5).to_string())

Trades: 10,071,946
1-second bars: 526,551
Mean trades per bar: 19.1
Bars with any buy: 79.2%
Bars with any sell: 79.5%

Sample:
      timestamp       mid  best_bid  best_ask  bar_high  bar_low  bar_volume  bar_trades  toxic_rate any_buy  max_buy_price  buy_volume  buy_trades any_sell  min_sell_price  sell_volume  sell_trades
0  1.725840e+09  54849.95   54849.9   54850.0   54850.0  54849.9       0.074           2         0.0    True        54850.0       0.050         1.0     True         54849.9        0.024          1.0
1  1.725840e+09  54849.95   54849.9   54850.0   54852.0  54850.0       2.658          29         0.0    True        54852.0       0.175        16.0     True         54851.6        2.483         13.0
2  1.725840e+09  54851.95   54851.9   54852.0   54852.0  54851.9       0.053           4         0.0    True        54852.0       0.013         2.0     True         54851.9        0.040          2.0
3  1.725840e+09  54851.95   54851.9   54852.0   54852.0  54851.9       0.131

In [80]:
def estimate_parameters_bars(bars_df: pd.DataFrame, 
                              bar_seconds: int,
                              asset: str, 
                              week: str) -> dict:
    """
    Estimate σ and κ from bar-level data.
    
    σ: std of log mid-price returns per bar
    κ: mean trades per bar (order arrival intensity at bar frequency)
    """
    # σ per bar
    log_returns = np.log(
        bars_df['mid'] / bars_df['mid'].shift(1)
    ).dropna()
    sigma_per_bar = log_returns.std()
    sigma_price_per_bar = sigma_per_bar * bars_df['mid'].mean()
    
    # κ: trades per bar
    kappa = bars_df['bar_trades'].mean()
    
    # Annualised σ for reference
    bars_per_year = 365 * 24 * 3600 / bar_seconds
    sigma_annualised = sigma_per_bar * np.sqrt(bars_per_year)
    
    print(f"\n{asset} {week} ({bar_seconds}s bars):")
    print(f"  n_bars:          {len(bars_df):,}")
    print(f"  κ (trades/bar):  {kappa:.2f}")
    print(f"  σ (per bar):     {sigma_per_bar:.8f}")
    print(f"  σ (annualised):  {sigma_annualised:.2%}")
    
    return {
        'asset': asset,
        'week': week,
        'kappa': kappa,
        'sigma_per_bar': sigma_per_bar,
        'sigma_price_per_bar': sigma_price_per_bar,
        'sigma_annualised': sigma_annualised,
        'bar_seconds': bar_seconds,
        'n_bars': len(bars_df)
    }
    
def run_backtest_bars(bars_df: pd.DataFrame,
                      gamma: float,
                      sigma: float,
                      kappa: float,
                      T_seconds: float = 86400.0,
                      bar_seconds: int = 1) -> tuple[MarketMakerState, pd.DataFrame]:
    """
    Bar-level A-S market-making backtest.
    
    MM posts quotes once per bar. Filled if any trade in the bar
    crosses the quote and MM is competitive (Option B).
    """
    state = MarketMakerState()
    results = []
    
    t_start = bars_df['timestamp'].iloc[0]
    
    for _, bar in bars_df.iterrows():
        t = bar['timestamp']
        mid = bar['mid']
        best_bid = bar['best_bid']
        best_ask = bar['best_ask']
        
        # Time remaining in trading day
        t_of_day = (t - t_start) % T_seconds
        T_minus_t = max(T_seconds - t_of_day, float(bar_seconds))
        
        # Compute A-S quotes
        mm_bid, mm_ask, r, delta, spread_bps = avellaneda_stoikov_quotes_adjusted(
            s=mid, q=state.inventory,
            gamma=gamma, sigma=sigma,
            kappa=kappa, T_minus_t=T_minus_t,
            best_bid=best_bid, best_ask=best_ask
        )
        
        # Option B: competitive check
        mm_ask_competitive = mm_ask <= best_ask
        mm_bid_competitive = mm_bid >= best_bid
        
        filled_buy = False
        filled_sell = False
        fill_buy_price = np.nan
        fill_sell_price = np.nan
        
        # Check sell fill (buy aggressor lifts MM's ask)
        if (mm_ask_competitive and 
            bar['any_buy'] and 
            bar['max_buy_price'] >= mm_ask):
            filled_sell = True
            fill_sell_price = mm_ask
            fill_qty = bar['buy_volume'] / bar['buy_trades']  # avg trade size
            state.cash += fill_sell_price * fill_qty
            state.inventory -= fill_qty
        
        # Check buy fill (sell aggressor hits MM's bid)
        if (mm_bid_competitive and 
            bar['any_sell'] and 
            bar['min_sell_price'] <= mm_bid):
            filled_buy = True
            fill_buy_price = mm_bid
            fill_qty = bar['sell_volume'] / bar['sell_trades']
            state.cash -= fill_buy_price * fill_qty
            state.inventory += fill_qty
        
        # Mark-to-market PnL
        pnl = state.cash + state.inventory * mid
        
        results.append({
            'timestamp': t,
            'mid': mid,
            'mm_bid': mm_bid,
            'mm_ask': mm_ask,
            'reservation_price': r,
            'half_spread': delta,
            'spread_bps': spread_bps,
            'mm_ask_competitive': mm_ask_competitive,
            'mm_bid_competitive': mm_bid_competitive,
            'filled_buy': filled_buy,
            'filled_sell': filled_sell,
            'fill_buy_price': fill_buy_price,
            'fill_sell_price': fill_sell_price,
            'inventory': state.inventory,
            'cash': state.cash,
            'pnl': pnl,
            'toxic_rate': bar['toxic_rate'],
        })
    
    results_df = pd.DataFrame(results)
    return state, results_df

In [81]:
# Load per-asset-week DataFrames
all_data = {}
for asset in ASSETS:
    for week in WEEKS:
        all_data[(asset, week)] = load_backtest_data(data_dir, asset, week)

# Aggregate all asset-weeks to bars
bars_data = {}
for asset in ASSETS:
    for week in WEEKS:
        bars_data[(asset, week)] = aggregate_to_bars(
            all_data[(asset, week)], bar_seconds=1
        )

# Estimate parameters at bar level
bar_params = {}
for asset in ASSETS:
    for week in WEEKS:
        bar_params[(asset, week)] = estimate_parameters_bars(
            bars_data[(asset, week)], 
            bar_seconds=1,
            asset=asset, 
            week=week
        )

# Display summary
bar_params_df = pd.DataFrame(bar_params.values())
print("\n--- Bar-Level Parameter Summary ---")
print(bar_params_df[['asset', 'week', 'kappa', 'sigma_per_bar', 
                       'sigma_annualised']].to_string(index=False))


BTCUSDT week1 (1s bars):
  n_bars:          526,551
  κ (trades/bar):  19.13
  σ (per bar):     0.00007937
  σ (annualised):  44.57%

BTCUSDT week2 (1s bars):
  n_bars:          525,290
  κ (trades/bar):  22.91
  σ (per bar):     0.00007091
  σ (annualised):  39.82%

BTCUSDT week3 (1s bars):
  n_bars:          558,416
  κ (trades/bar):  43.43
  σ (per bar):     0.00013122
  σ (annualised):  73.69%

ETHUSDT week1 (1s bars):
  n_bars:          379,955
  κ (trades/bar):  11.99
  σ (per bar):     0.00011170
  σ (annualised):  62.73%

ETHUSDT week2 (1s bars):
  n_bars:          415,317
  κ (trades/bar):  17.19
  σ (per bar):     0.00010550
  σ (annualised):  59.25%

ETHUSDT week3 (1s bars):
  n_bars:          319,539
  κ (trades/bar):  31.46
  σ (per bar):     0.00019477
  σ (annualised):  109.38%

SOLUSDT week1 (1s bars):
  n_bars:          409,128
  κ (trades/bar):  9.70
  σ (per bar):     0.00012016
  σ (annualised):  67.48%

SOLUSDT week2 (1s bars):
  n_bars:          428,987
  κ (trad

In [82]:
def calibrate_gamma_bars(bars_df, sigma, kappa, gamma_grid, 
                          bar_seconds=1, T_seconds=86400.0):
    results = []
    for gamma in gamma_grid:
        print(f"  Testing γ={gamma}...", end=' ')
        _, bt = run_backtest_bars(
            bars_df=bars_df,
            gamma=gamma,
            sigma=sigma,
            kappa=kappa,
            T_seconds=T_seconds,
            bar_seconds=bar_seconds
        )
        metrics = compute_backtest_metrics(bt)
        metrics['gamma'] = gamma
        results.append(metrics)
        print(f"inv_95th={metrics['inventory_95th']:.3f}, "
              f"sharpe={metrics['sharpe']:.3f}, "
              f"competitive_rate={metrics['competitive_rate']:.4f}, "
              f"fill_rate={metrics['fill_rate']:.4f}, "
              f"mean_spread_bps={metrics['mean_spread_bps']:.4f}")
    return pd.DataFrame(results)

gamma_grid = [0.0001, 0.0005, 0.001, 0.005, 0.01, 0.05, 0.1]

btc_w1_bar_params = bar_params[('BTCUSDT', 'week1')]

print("Calibrating γ on BTC week 1 (1s bars)...")
cal_results = calibrate_gamma_bars(
    bars_df=bars_data[('BTCUSDT', 'week1')],
    sigma=btc_w1_bar_params['sigma_per_bar'],
    kappa=btc_w1_bar_params['kappa'],
    gamma_grid=gamma_grid
)

print("\n--- Calibration Results ---")
print(cal_results[['gamma', 'inventory_95th', 'inventory_max',
                    'sharpe', 'fill_rate', 'competitive_rate',
                    'mean_spread_bps']].to_string(index=False))

Calibrating γ on BTC week 1 (1s bars)...
  Testing γ=0.0001... inv_95th=65.492, sharpe=-10.635, competitive_rate=0.7860, fill_rate=0.6281, mean_spread_bps=0.0176
  Testing γ=0.0005... inv_95th=65.492, sharpe=-10.635, competitive_rate=0.7860, fill_rate=0.6281, mean_spread_bps=0.0176
  Testing γ=0.001... inv_95th=65.492, sharpe=-10.635, competitive_rate=0.7860, fill_rate=0.6281, mean_spread_bps=0.0176
  Testing γ=0.005... inv_95th=65.492, sharpe=-10.635, competitive_rate=0.7860, fill_rate=0.6281, mean_spread_bps=0.0176
  Testing γ=0.01... inv_95th=65.492, sharpe=-10.635, competitive_rate=0.7860, fill_rate=0.6281, mean_spread_bps=0.0176
  Testing γ=0.05... inv_95th=65.492, sharpe=-10.635, competitive_rate=0.7860, fill_rate=0.6281, mean_spread_bps=0.0176
  Testing γ=0.1... inv_95th=65.492, sharpe=-10.635, competitive_rate=0.7860, fill_rate=0.6281, mean_spread_bps=0.0176

--- Calibration Results ---
 gamma  inventory_95th  inventory_max     sharpe  fill_rate  competitive_rate  mean_spread_b

In [85]:
s = btc_bars['mid'].iloc[0]
best_bid = btc_bars['best_bid'].iloc[0]
best_ask = btc_bars['best_ask'].iloc[0]
sigma = btc_w1_bar_params['sigma_per_bar']
kappa = btc_w1_bar_params['kappa']

print(f"{'gamma':>8} {'q':>8} {'r':>12} {'delta_log':>12} {'mm_bid':>12} {'mm_ask':>12} {'bid_comp':>10}")
for gamma in [0.0001, 0.001, 0.01, 0.1]:
    for q in [0, 1, 10, 50, 100]:
        mm_bid, mm_ask, r, delta_log, spread_bps = avellaneda_stoikov_quotes_adjusted(
            s=s, q=q, gamma=gamma, sigma=sigma,
            kappa=kappa, T_minus_t=43200.0,
            best_bid=best_bid, best_ask=best_ask
        )
        bid_comp = mm_bid >= best_bid
        print(f"{gamma:>8.4f} {q:>8} {r:>12.4f} {delta_log:>12.8f} {mm_bid:>12.4f} {mm_ask:>12.4f} {str(bid_comp):>10}")
    print()

   gamma        q            r    delta_log       mm_bid       mm_ask   bid_comp
  0.0001        0   54849.9500   0.10455770   54849.9000   54850.0000      False
  0.0001        1   54849.9500   0.10455770   54849.9000   54850.0000      False
  0.0001       10   54849.9500   0.10455770   54849.9000   54850.0000      False
  0.0001       50   54849.9500   0.10455770   54849.9000   54850.0000      False
  0.0001      100   54849.9500   0.10455770   54849.9000   54850.0000      False

  0.0010        0   54849.9500   0.10455549   54849.9000   54850.0000      False
  0.0010        1   54849.9500   0.10455549   54849.9000   54850.0000      False
  0.0010       10   54849.9500   0.10455549   54849.9000   54850.0000      False
  0.0010       50   54849.9500   0.10455549   54849.9000   54850.0000      False
  0.0010      100   54849.9500   0.10455549   54849.9000   54850.0000      False

  0.0100        0   54849.9500   0.10453335   54849.9000   54850.0000      False
  0.0100        1   54849.